### Tyendinaga Data
##### Notes:
- data exported from Grilla
- using data with filename Tyendinaga1_001.asc as main data source to make plots with
- Tyendinaga1_001.asc data was recorded on Oct. 26, 2025 from 18:07:12 to 18:27:12

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal
import scipy.stats as ss
from sklearn.preprocessing import MinMaxScaler
import scipy.fft
from scipy.signal.windows import hann
from scipy import interpolate
from scipy.ndimage import uniform_filter
from scipy.signal import ShortTimeFFT
from scipy.signal import savgol_filter
from scipy.signal import hilbert
from scipy.signal import welch
import subprocess
import os

In [ ]:
# list of the library versions used:
print("Version Notes:")
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)

##### DataFrame --> File Name Legend:
- df --> Tyendinaga_001.asc
- df2 --> Tyendinaga2_001.asc
- df3 --> Tyendinaga3_001.asc
- df4 --> Tyendinaga4_001.asc
- df5 --> Tyendinaga5_001.asc

In [ ]:
# Filename: Tyendinaga1
column_names = ['NS', 'EW', 'Z', 'nsL', 'ewL', 'zL', 'aY', 'aX', 'aZ']
df = pd.read_table('Tyendinaga1_001.asc', delimiter=r'\s+', encoding='latin-1', skiprows=32, names=column_names)
#df.head().style

In [ ]:
# Filename: Tyendinaga2
# Loading the rest of the Tyendinaga files so can make comparitive plots between datasets
column_names2 = ['NS', 'EW', 'Z', 'nsL', 'ewL', 'zL', 'aY', 'aX', 'aZ']
df2 = pd.read_table('Tyendinaga2_001.asc', delimiter=r'\s+', encoding='latin-1', skiprows=32, names=column_names2)
#df2.head().style

In [ ]:
# Filename: Tyendinaga3
column_names3 = ['NS', 'EW', 'Z', 'nsL', 'ewL', 'zL', 'aY', 'aX', 'aZ']
df3 = pd.read_table('Tyendinaga3_001.asc', delimiter=r'\s+', encoding='latin-1', skiprows=32, names=column_names3)
#df3.head().style

In [ ]:
# Filename: Tyendinaga4
column_names4 = ['NS', 'EW', 'Z', 'nsL', 'ewL', 'zL', 'aY', 'aX', 'aZ']
df4 = pd.read_table('Tyendinaga4_001.asc', delimiter=r'\s+', encoding='latin-1', skiprows=32, names=column_names4)
#df4.head().style

In [ ]:
# Filename: Tyendinaga5
column_names5 = ['NS', 'EW', 'Z', 'nsL', 'ewL', 'zL', 'aY', 'aX', 'aZ']
df5 = pd.read_table('Tyendinaga5_001.asc', delimiter=r'\s+', encoding='latin-1', skiprows=32, names=column_names5)
#df5.head().style

##### Check for null values:
- will show "False" if no null values

In [ ]:
any_null = df.isnull().any().any()
print("df: ",any_null) # Will show false if no null values
#df.info()
print("df2:", df2.isnull().any().any())
print("df3:", df3.isnull().any().any())
print("df4:", df4.isnull().any().any())
print("df5:", df5.isnull().any().any())

##### Columns of Interest:
- NS: North-South component of motion
- EW: East-West component of motion
- Z: Vertical (Z-axis) component of motion

##### Other Notes:
- Sampling rate for all files: 1024 Hz
- See below for preview of the data:

In [ ]:
#print(df.columns)
# Want to make a time column:
sampling_rate = 1024  # Hz (same for all data files)
dataframes = [df, df2, df3, df4, df5]
for i, dataframe in enumerate(dataframes):
    if i==0:
        print("dataframe: df")
    else:
        print(f"dataframe: df{i+1}")
    time = dataframe.index/sampling_rate
    dataframe['time (s)'] = time
    print(dataframe.head())
    print("")

***
### Plotting the Time Series
##### Notes:
- Plot the NS, EW, and Z data separately

In [ ]:
# reminder: dataframes = [df, df2, df3, df4, df5]
for i, dframe in enumerate(dataframes):
    if i==0:
        print('dataframe: df')
    else:
        print(f'dataframe: df{i+1}')

    # Plot:
    fig, axes = plt.subplots(3, 1, figsize=(10, 10))
    time_minutes = dframe['time (s)']/60
    # North-South data:
    axes[0].plot(time_minutes, dframe['NS'], label='North-South', color='forestgreen')
    axes[0].set_ylabel('North-South Component')
    axes[0].set_title('North-South Data')
    axes[0].set_xlabel('Time (min)')
    axes[0].grid(True)
    
    # East-West data:
    axes[1].plot(time_minutes, dframe['EW'], label='East-West', color='rebeccapurple')
    axes[1].set_ylabel('East-West Component')
    axes[1].set_title('East-West Data')
    axes[1].set_xlabel('Time (min)')
    axes[1].grid(True)
    
    # Z data:
    axes[2].plot(time_minutes, dframe['Z'], label='Z/Vertical Component', color='royalblue')
    axes[2].set_ylabel('Z Component')
    axes[2].set_title('Z-Component Data')
    axes[2].set_xlabel('Time (min)')
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()

***
### Normalizing, Detrending and Shifting Data
##### Notes:
- data range of time series was normalized using MinMaxScaler from sklearn
- data was linearly detrended using signal.detrend from scipy
- the mean of the data was shifted to zero

In [ ]:
# Linearly detrend data, normalize and shift to zero mean:
# reminder: dataframes = [df, df2, df3, df4, df5]
for i, dframe in enumerate(dataframes):
    if i==0:
        print('dataframe: df')
    else:
        print(f'dataframe: df{i+1}')
        
    # Detrend
    dframe['NS_Detrended'] = signal.detrend(dframe['NS'], type='linear') # North-South data
    dframe['EW_Detrended'] = signal.detrend(dframe['EW'], type='linear') # East-West data
    dframe['Z_Detrended'] = signal.detrend(dframe['Z'], type='linear') # Z-component data

    # Normalize
    scaler = MinMaxScaler()
    dframe['NS_Normalized'] = scaler.fit_transform(dframe['NS_Detrended'].values.reshape(-1, 1)).flatten() # North-South
    dframe['EW_Normalized'] = scaler.fit_transform(dframe['EW_Detrended'].values.reshape(-1, 1)).flatten() # East-West
    dframe['Z_Normalized'] = scaler.fit_transform(dframe['Z_Detrended'].values.reshape(-1, 1)).flatten() # Z-component
    print(f"NS Min-Max normalized range: [{dframe['NS_Normalized'].min():.3f}, {dframe['NS_Normalized'].max():.3f}]")
    print(f"EW Min-Max normalized range: [{dframe['EW_Normalized'].min():.3f}, {dframe['EW_Normalized'].max():.3f}]")
    print(f"Z Min-Max normalized range: [{dframe['Z_Normalized'].min():.3f}, {dframe['Z_Normalized'].max():.3f}]")

    # Shift Mean:
    dframe['NS_Normalized_Shifted'] = (dframe['NS_Normalized'] - np.mean(dframe['NS_Normalized']))
    dframe['EW_Normalized_Shifted'] = (dframe['EW_Normalized'] - np.mean(dframe['EW_Normalized']))
    dframe['Z_Normalized_Shifted'] = (dframe['Z_Normalized'] - np.mean(dframe['Z_Normalized']))
    print(f"New range for NS: [{dframe['NS_Normalized_Shifted'].min():.3f}, {dframe['NS_Normalized_Shifted'].max():.3f}]")
    print(f"New mean for NS: {dframe['NS_Normalized_Shifted'].mean():.6f}")
    print(f"New range for EW: [{dframe['EW_Normalized_Shifted'].min():.3f}, {dframe['EW_Normalized_Shifted'].max():.3f}]")
    print(f"New mean for EW: {dframe['EW_Normalized_Shifted'].mean():.6f}")
    print(f"New range for Z: [{dframe['Z_Normalized_Shifted'].min():.3f}, {dframe['Z_Normalized_Shifted'].max():.3f}]")
    print(f"New mean for Z: {dframe['Z_Normalized_Shifted'].mean():.6f}")

    # Plot:
    fig, axes = plt.subplots(3, 1, figsize=(10, 10))
    time_minutes = dframe['time (s)']/60
    # North-South data:
    axes[0].plot(time_minutes, dframe['NS_Normalized_Shifted'], label='North-South', color='forestgreen')
    axes[0].set_ylabel('North-South Component')
    axes[0].set_title('DNS North-South Data')
    axes[0].set_xlabel('Time (min)')
    axes[0].grid(True)

    # East-West data:
    axes[1].plot(time_minutes, dframe['EW_Normalized_Shifted'], label='East-West', color='rebeccapurple')
    axes[1].set_ylabel('East-West Component')
    axes[1].set_title('DNS East-West Data')
    axes[1].set_xlabel('Time (min)')
    axes[1].grid(True)
    
    # Z data:
    axes[2].plot(time_minutes, dframe['Z_Normalized_Shifted'], label='Z/Vertical Component', color='royalblue')
    axes[2].set_ylabel('Z Component')
    axes[2].set_title('DNS Z-Component Data')
    axes[2].set_xlabel('Time (min)')
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()
    print("DNS = Detrended, Normalized, Shifted")
    print("")
    # Stats for plots
    print("North-South Stats:")
    print(ss.describe(dframe['NS_Normalized_Shifted']))
    print("median:", dframe["NS_Normalized_Shifted"].median())
    print("mode:", ss.mode(df["NS_Normalized_Shifted"])) 
    print("standard deviation:", dframe["NS_Normalized_Shifted"].std())
    print("")
    print("East-West Stats:")
    print(ss.describe(dframe['EW_Normalized_Shifted']))
    print("median:", dframe["EW_Normalized_Shifted"].median())
    print("mode:", ss.mode(dframe["EW_Normalized_Shifted"])) 
    print("standard deviation:", dframe["EW_Normalized_Shifted"].std())
    print("")
    print("Z-component Stats:")
    print(ss.describe(dframe['Z_Normalized_Shifted']))
    print("median:", dframe["Z_Normalized_Shifted"].median())
    print("mode:", ss.mode(dframe["Z_Normalized_Shifted"])) 
    print("standard deviation:", dframe["Z_Normalized_Shifted"].std())
    print("~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~")

***
### Subsampling the Time Series
##### Notes:
- The original sampling rate for the data is 1024 Hz
- Different step sizes were applied to the normalized, detrended and mean-shifted (DNS) data for subsampling
    - Adjust decimation_factor as needed
- Adjust x-axis limits to focus in on certain times

In [ ]:
original_sampling_rate = 1024 # Hz
def subsample(data, decimation_factor, sr):
    """Subsample data using scipy.signal.decimate with anti-aliasing filter.
    data: input data (array)
    decimation_factor: factor by which to reduce sampling rate
    sr: Original sampling rate"""
    # scipy.signal.decimate applies an anti-aliasing filter automatically
    # Use a higher order filter for better anti-aliasing (default is 8)
    decimated_data = scipy.signal.decimate(data, decimation_factor, ftype='iir', zero_phase=True)  
    # get new sampling rate:
    new_sr = sr / decimation_factor
    #print(f"New sampling rate: {new_sr} Hz")
    return decimated_data, new_sr

# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Compute results for NS, EW, and Z data:
# (Using stepsize of 7 for decimation_factor)
# df:
NS_sub_results_df, NS_new_sr_df = subsample(df['NS_Normalized_Shifted'].values, 7, original_sampling_rate)
EW_sub_results_df, EW_new_sr_df = subsample(df['EW_Normalized_Shifted'].values, 7, original_sampling_rate)
Z_sub_results_df, Z_new_sr_df = subsample(df['Z_Normalized_Shifted'].values, 7, original_sampling_rate)
#df2:
NS_sub_results_df2, NS_new_sr_df2 = subsample(df2['NS_Normalized_Shifted'].values, 7, original_sampling_rate)
EW_sub_results_df2, EW_new_sr_df2 = subsample(df2['EW_Normalized_Shifted'].values, 7, original_sampling_rate)
Z_sub_results_df2, Z_new_sr_df2 = subsample(df2['Z_Normalized_Shifted'].values, 7, original_sampling_rate)
#df3:
NS_sub_results_df3, NS_new_sr_df3 = subsample(df3['NS_Normalized_Shifted'].values, 7, original_sampling_rate)
EW_sub_results_df3, EW_new_sr_df3 = subsample(df3['EW_Normalized_Shifted'].values, 7, original_sampling_rate)
Z_sub_results_df3, Z_new_sr_df3 = subsample(df3['Z_Normalized_Shifted'].values, 7, original_sampling_rate)
#df4:
NS_sub_results_df4, NS_new_sr_df4 = subsample(df4['NS_Normalized_Shifted'].values, 7, original_sampling_rate)
EW_sub_results_df4, EW_new_sr_df4 = subsample(df4['EW_Normalized_Shifted'].values, 7, original_sampling_rate)
Z_sub_results_df4, Z_new_sr_df4 = subsample(df4['Z_Normalized_Shifted'].values, 7, original_sampling_rate)
#df5:
NS_sub_results_df5, NS_new_sr_df5 = subsample(df5['NS_Normalized_Shifted'].values, 7, original_sampling_rate)
EW_sub_results_df5, EW_new_sr_df5 = subsample(df5['EW_Normalized_Shifted'].values, 7, original_sampling_rate)
Z_sub_results_df5, Z_new_sr_df5 = subsample(df5['Z_Normalized_Shifted'].values, 7, original_sampling_rate)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
NS_subsampled_data = [NS_sub_results_df, NS_sub_results_df2, NS_sub_results_df3, NS_sub_results_df4, NS_sub_results_df5]
EW_subsampled_data = [EW_sub_results_df, EW_sub_results_df2, EW_sub_results_df3, EW_sub_results_df4, EW_sub_results_df5]
Z_subsampled_data = [Z_sub_results_df, Z_sub_results_df2, Z_sub_results_df3, Z_sub_results_df4, Z_sub_results_df5]

for i, dframe in enumerate(dataframes):
    # Plot:
    time_minutes = dframe['time (s)']/60
    # North-South data:
    plt.figure(figsize=(12,4))
    plt.plot(time_minutes, dframe['NS_Normalized_Shifted'], label='North-South (Full)', color='red')
    plt.plot(time_minutes[::7], NS_subsampled_data[i], label='North-South (Step=7)', color='forestgreen')
    plt.ylabel('North-South Component')
    plt.title('Subsampled and Full DNS North-South Data')
    plt.xlabel('Time (min)')
    plt.legend(loc='upper left')
    plt.grid(True)
    
    # East-West data:
    plt.figure(figsize=(12,4))
    plt.plot(time_minutes, dframe['EW_Normalized_Shifted'], label='East-West (Full)', color='red')
    plt.plot(time_minutes[::7], EW_subsampled_data[i], label='East-West (Step=7)', color='rebeccapurple')
    plt.ylabel('East-West Component')
    plt.title('Subsampled and Full DNS East-West Data')
    plt.xlabel('Time (min)')
    plt.legend(loc='upper left')
    plt.grid(True)
    
    # Z data:
    plt.figure(figsize=(12,4))
    plt.plot(time_minutes, dframe['Z_Normalized_Shifted'], label='Z (Full)', color='red')
    plt.plot(time_minutes[::7], Z_subsampled_data[i], label='Z (Step=7)', color='royalblue')
    plt.ylabel('Z Component')
    plt.title('Subsampled and Full DNS Z-Component Data')
    plt.xlabel('Time (min)')
    plt.legend(loc='upper left')
    plt.grid(True)

plt.tight_layout()
plt.show()
# Note: New sampling rate is 146.285714... Hz using decimation_factor of 7

***
### Tapering the Time Series
##### Notes:
- A Hann window/raised cosine was used to taper the DNS time series
- The data is tapered starting at 10% from the edges (this percentage can be adjusted)

In [ ]:
# reminder (defined in previous cell): dataframes = [df, df2, df3, df4, df5]
def apply_hann_window(data, NS_column, EW_column, Z_column, taper_fraction = 0.1):
    """Taper edges of time series with Hann window
    taper_fraction = 0.1 will start taper at 10% from edges"""
    n= len(data)
    taper_length = int(n * taper_fraction) # should be same for all components

    # NS Data:
    # Create a copy of the data
    NS_windowed_data = data[NS_column].copy()
    NS_windowed_data = NS_windowed_data.astype(float) # needed for FFT code to work properly later on
    # Apply Hann window to the left edge
    NS_left_window = np.hanning(2 * taper_length)[:taper_length]
    NS_windowed_data.iloc[:taper_length] *= NS_left_window
    # Apply Hann window to the right edge
    NS_right_window = np.hanning(2 * taper_length)[taper_length:]
    NS_windowed_data.iloc[-taper_length:] *= NS_right_window

    # EW Data:
    # Create a copy of the data
    EW_windowed_data = data[EW_column].copy()
    EW_windowed_data = EW_windowed_data.astype(float) 
    # Apply Hann window to the left edge
    EW_left_window = np.hanning(2 * taper_length)[:taper_length]
    EW_windowed_data.iloc[:taper_length] *= EW_left_window
    # Apply Hann window to the right edge
    EW_right_window = np.hanning(2 * taper_length)[taper_length:]
    EW_windowed_data.iloc[-taper_length:] *= EW_right_window

    # Z Data:
    Z_windowed_data = data[Z_column].copy()
    Z_windowed_data = Z_windowed_data.astype(float) 
    # Apply Hann window to the left edge
    Z_left_window = np.hanning(2 * taper_length)[:taper_length]
    Z_windowed_data.iloc[:taper_length] *= Z_left_window
    # Apply Hann window to the right edge
    Z_right_window = np.hanning(2 * taper_length)[taper_length:]
    Z_windowed_data.iloc[-taper_length:] *= Z_right_window
    
    return NS_windowed_data, EW_windowed_data, Z_windowed_data
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# df:
NS_hann_result_df, EW_hann_result_df, Z_hann_result_df = apply_hann_window(df, 'NS_Normalized_Shifted', 'EW_Normalized_Shifted', 'Z_Normalized_Shifted')
# df2:
NS_hann_result_df2, EW_hann_result_df2, Z_hann_result_df2 = apply_hann_window(df2, 'NS_Normalized_Shifted', 'EW_Normalized_Shifted', 'Z_Normalized_Shifted')
# df3:
NS_hann_result_df3, EW_hann_result_df3, Z_hann_result_df3 = apply_hann_window(df3, 'NS_Normalized_Shifted', 'EW_Normalized_Shifted', 'Z_Normalized_Shifted')
# df4:
NS_hann_result_df4, EW_hann_result_df4, Z_hann_result_df4 = apply_hann_window(df4, 'NS_Normalized_Shifted', 'EW_Normalized_Shifted', 'Z_Normalized_Shifted')
# df5:
NS_hann_result_df5, EW_hann_result_df5, Z_hann_result_df5 = apply_hann_window(df5, 'NS_Normalized_Shifted', 'EW_Normalized_Shifted', 'Z_Normalized_Shifted')
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
NS_hann_results = [NS_hann_result_df, NS_hann_result_df2, NS_hann_result_df3, NS_hann_result_df4, NS_hann_result_df5]
EW_hann_results = [EW_hann_result_df, EW_hann_result_df2, EW_hann_result_df3, EW_hann_result_df4, EW_hann_result_df5]
Z_hann_results = [Z_hann_result_df, Z_hann_result_df2, Z_hann_result_df3, Z_hann_result_df4, Z_hann_result_df5]

# Plot tapered data with full data for all dataframes:
for i, dframe in enumerate(dataframes):
    if i == 0:
        print("dataframe: df")
    else:
        print(f"dataframe: df{i+1}")
    # Plot:
    fig, axes = plt.subplots(3, 1, figsize=(10, 10))
    time_minutes = dframe['time (s)']/60
    # North-South data:
    axes[0].plot(time_minutes, dframe['NS_Normalized_Shifted'], label='North-South (Full)', color='red')
    axes[0].plot(time_minutes, NS_hann_results[i], label='North-South (Tapered)', color='forestgreen')
    axes[0].set_ylabel('North-South Component')
    axes[0].set_title('Tapered and Full DNS North-South Data')
    axes[0].set_xlabel('Time (min)')
    axes[0].legend()
    axes[0].grid(True)
    
    # East-West data:
    axes[1].plot(time_minutes, dframe['EW_Normalized_Shifted'], label='East-West (Full)', color='red')
    axes[1].plot(time_minutes, EW_hann_results[i], label='East-West (Tapered)', color='rebeccapurple')
    axes[1].set_ylabel('East-West Component')
    axes[1].set_title('Tapered and Full DNS East-West Data')
    axes[1].set_xlabel('Time (min)')
    axes[1].legend()
    axes[1].grid(True)
    
    # Z data:
    axes[2].plot(time_minutes, dframe['Z_Normalized_Shifted'], label='Z (Full)', color='red')
    axes[2].plot(time_minutes, Z_hann_results[i], label='Z (Tapered)', color='royalblue')
    axes[2].set_ylabel('Z Component')
    axes[2].set_title('Tapered and Full DNS Z-Component Data')
    axes[2].set_xlabel('Time (min)')
    axes[2].legend()
    #axes[2].set_xlim(-0.001,0.01)
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()

***
### Combining Subsampling and Tapering
##### Notes:
- The full series was tapered before subsampling
- A Hann window was used for tapering
- A consistent step size is used for subsampling across the North-South, East-West, and Up-Down (Z) directions
    - adjust the step size used for subsampling as needed

In [ ]:
# Put tapered results into subsample function from a previous cell:
def subsample_windowed(NS_windowed, EW_windowed, Z_windowed, sr=1024, decimation_factor=7):
    NS_decimated_data = scipy.signal.decimate(NS_windowed, decimation_factor, ftype='iir', zero_phase=True)
    EW_decimated_data = scipy.signal.decimate(EW_windowed, decimation_factor, ftype='iir', zero_phase=True)
    Z_decimated_data = scipy.signal.decimate(Z_windowed, decimation_factor, ftype='iir', zero_phase=True)
    new_sr = sr/decimation_factor
    return NS_decimated_data, EW_decimated_data, Z_decimated_data, new_sr
# Results for all dataframes:  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# df:
NS_decimated_data_df, EW_decimated_data_df, Z_decimated_data_df, new_sr_df = subsample_windowed(NS_hann_result_df, EW_hann_result_df, Z_hann_result_df)
# df2:
NS_decimated_data_df2, EW_decimated_data_df2, Z_decimated_data_df2, new_sr_df2 = subsample_windowed(NS_hann_result_df2, EW_hann_result_df2, Z_hann_result_df2)
# df3:
NS_decimated_data_df3, EW_decimated_data_df3, Z_decimated_data_df3, new_sr_df3 = subsample_windowed(NS_hann_result_df3, EW_hann_result_df3, Z_hann_result_df3)
# df4:
NS_decimated_data_df4, EW_decimated_data_df4, Z_decimated_data_df4, new_sr_df4 = subsample_windowed(NS_hann_result_df4, EW_hann_result_df4, Z_hann_result_df4)
# df5:
NS_decimated_data_df5, EW_decimated_data_df5, Z_decimated_data_df5, new_sr_df5 = subsample_windowed(NS_hann_result_df5, EW_hann_result_df5, Z_hann_result_df5)
# ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
NS_taper_sub = [NS_decimated_data_df, NS_decimated_data_df2, NS_decimated_data_df3, NS_decimated_data_df4, NS_decimated_data_df5]
EW_taper_sub = [EW_decimated_data_df, EW_decimated_data_df2, EW_decimated_data_df3, EW_decimated_data_df4, EW_decimated_data_df5]
Z_taper_sub = [Z_decimated_data_df, Z_decimated_data_df2, Z_decimated_data_df3, Z_decimated_data_df4, Z_decimated_data_df5]

for i, dframe in enumerate(dataframes):
    if i ==0:
        print("dataframe: df")
    else:
        print(f"dataframe: df{i+1}")
    # Plot:
    fig, axes = plt.subplots(3, 1, figsize=(12, 12))
    time_minutes = dframe['time (s)']/60
    # North-South data:
    axes[0].plot(time_minutes, dframe['NS_Normalized_Shifted'], label='North-South (Full)', color='red')
    axes[0].plot(time_minutes[::7], NS_taper_sub[i], label='Tapered North-South (Step=7)', color='forestgreen') 
    # NOTE: make sure step for time_minutes=decimation_factor
    axes[0].set_ylabel('North-South Component')
    axes[0].set_title('Tapered/Subsampled and Full DNS North-South Data')
    axes[0].set_xlabel('Time (min)')
    axes[0].legend()
    axes[0].grid(True)
    
    # East-West data:
    axes[1].plot(time_minutes, dframe['EW_Normalized_Shifted'], label='East-West (Full)', color='red')
    axes[1].plot(time_minutes[::7], EW_taper_sub[i], label='Tapered East-West (Step=7)', color='rebeccapurple')
    axes[1].set_ylabel('East-West Component')
    axes[1].set_title('Tapered/Subsampled and Full DNS East-West Data')
    axes[1].set_xlabel('Time (min)')
    axes[1].legend()
    axes[1].grid(True)
    
    # Z data:
    axes[2].plot(time_minutes, dframe['Z_Normalized_Shifted'], label='Z (Full)', color='red')
    axes[2].plot(time_minutes[::7], Z_taper_sub[i], label='Tapered Z (Step=7)', color='royalblue')
    axes[2].set_ylabel('Z Component')
    axes[2].set_title('Tapered/Subsampled and Full DNS Z-Component Data')
    axes[2].set_xlabel('Time (min)')
    axes[2].legend()
    #axes[2].set_xlim(-0.001,0.01)
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()

***
### Fast Fourier Transform
- Updated from original: Decreased frequency resolution
- Defined a coherent gain function in attempt to correct for amplitude reduction from windowing

##### Coherent gain equation:
$G_c = \frac{1}{N}\sum_{n=0}^{N-1}\omega[n]$

- $G_c$: coherent gain
- $\omega[n]$: window
- $N$: window length

In [ ]:
# Define helper functions:
# Hann window one column at a time:
def apply_hann_window1(data, column, taper_fraction = 0.1):
    """Taper edges of time series with Hann window
    taper_fraction = 0.1 will start taper at 10% from edges"""
    n= len(data)
    taper_length = int(n * taper_fraction)
    
    # Create a copy of the data
    windowed_data = data[column].copy()
    windowed_data = windowed_data.astype(float) # needed for FFT code to work properly
    
    # Apply Hann window to the left edge
    left_window = np.hanning(2 * taper_length)[:taper_length]
    windowed_data[:taper_length] *= left_window
    
    # Apply Hann window to the right edge
    right_window = np.hanning(2 * taper_length)[taper_length:]
    windowed_data[-taper_length:] *= right_window
    
    return windowed_data

#print(type(apply_hann_window1(df, "NS_Normalized_Shifted")))

def calculate_coherent_gain(data, taper_fraction=0.1):
    """Calculate coherent gain for partial Hann taper"""
    n=len(data)
    taper_length = int(n * taper_fraction)
    
    # Create the full window
    window = np.ones(n)
    
    # Left taper
    left_window = np.hanning(2 * taper_length)[:taper_length]
    window[:taper_length] = left_window
    
    # Right taper
    right_window = np.hanning(2 * taper_length)[taper_length:]
    window[-taper_length:] = right_window
    
    # Coherent gain is the mean
    coherent_gain = np.mean(window)
    
    return coherent_gain

##### FFT With Frequency Bin Averaging:

In [ ]:
def plot_averaged_fft(data, column, sr, segment_size, title=''):
    """Plot FFT with frequency bin averaging 
    data: pandas DataFrame
    column: relevant data column from data, string
    sr: Sampling rate in Hz
    segment_size:segment size for averaging, int
    title: plot title, str"""
    # Goal: decrease frequency resolution of the FFT
    # Signal parameters
    T = 1 / sr  # Sampling period
    # signal x:
    x = data[column].values
    N = len(x)
    
    # Parameters for averaging
    overlap = 0.5   # 50% overlap
    step_size = int(segment_size * (1 - overlap))
    
    # Calculate number of segments
    num_segments = (N - segment_size) // step_size + 1
    
    # Initialize array to store summed FFT results
    avg_fft_magnitude = np.zeros(segment_size // 2 + 1) # only need one side for real input
    
    for i in range(num_segments):
        start = i * step_size
        end = start + segment_size
        segment = x[start:end]
        
        # apply a Hanning window to the segment to reduce spectral leakage
        window = np.hanning(segment_size)
        windowed_segment = segment * window
        
        # Compute FFT of the segment
        segment_fft = scipy.fft.rfft(windowed_segment, segment_size)
        
        # Get the magnitude of the positive frequency side
        #segment_magnitude = np.abs(segment_fft)[:segment_size // 2 + 1]
        segment_magnitude = np.abs(segment_fft)[:segment_size // 2 + 1] / np.sum(window) # window sum is for amplitude correction
        
        # Accumulate the magnitudes
        #avg_fft_magnitude += segment_magnitude 
        avg_fft_magnitude += segment_magnitude ** 2 #accumulate power
    
    # Calculate the mean magnitude
    #avg_fft_magnitude /= num_segments
    avg_fft_magnitude = np.sqrt(avg_fft_magnitude / num_segments) # don't take square root if want power spectral density instead
    
    # Get the frequency bins
    freqs = scipy.fft.rfftfreq(segment_size, T)[:segment_size // 2 + 1]
    
    # Plot the averaged frequency spectrum
    plt.figure(figsize=(12, 6))
    #plt.plot(freqs, avg_fft_magnitude)
    plt.loglog(freqs, avg_fft_magnitude)
    plt.xlabel('Frequency (Hz)', fontsize=12)
    plt.ylabel('Averaged FFT Magnitude', fontsize=12)
    plt.title(title)
    plt.grid(True, which='both', alpha=0.5)
    plt.show()

    return freqs, avg_fft_magnitude
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
result1 = plot_averaged_fft(df, 'NS_Normalized_Shifted', sr=original_sampling_rate, segment_size=1000, title='North-South Averaged Frequency Spectrum')
result2 = plot_averaged_fft(df, 'EW_Normalized_Shifted', sr=original_sampling_rate, segment_size=1000, title='East-West Averaged Frequency Spectrum')
result3 = plot_averaged_fft(df, 'Z_Normalized_Shifted', sr=original_sampling_rate, segment_size=1000, title='Up-Down Averaged Frequency Spectrum')

##### Power Spectral Density With Welch's Method:
- Change the scaling parameter in welch() from 'density' to 'spectrum' to plot a squared magnitude spectrum instead

In [ ]:
# Trying power spectral density:
def welch_psd(data, column, sr, title, segment_size, window='hann'):
    """data: pandas DataFrame
    column: relevant data column from data, str
    sr: Sampling rate in Hz
    title: title for PSD plot, str
    segment_size: length of each segment
    window: desired window to use"""
    
    overlap = segment_size//2 # number of points to overlap between segments, here is 50% overlap
    freqs, psd = welch(data[column].values, fs=sr, nperseg=segment_size, noverlap=overlap, window=window, scaling='density')

    # Plot:
    plt.figure(figsize=(12,6))
    #plt.plot(freqs,psd)
    plt.loglog(freqs, psd)
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("PSD")
    plt.title(title)
    plt.grid(True, which='both', alpha=0.5)
    plt.ylim(bottom=10**-9)
    plt.show()

    return freqs, psd

# Results using normalized (DNS) data:
freqs_NS_, psd_NS_ = welch_psd(df, 'NS_Normalized_Shifted', original_sampling_rate, 'North-South PSD', segment_size=4096)
freqs_EW_, psd_EW_ = welch_psd(df, 'EW_Normalized_Shifted', original_sampling_rate, 'East-West PSD', segment_size=4096)
freqs_Z_, psd_Z_ = welch_psd(df, 'Z_Normalized_Shifted', original_sampling_rate, 'Up-Down PSD', segment_size=4096)

In [ ]:
# PSD plot for all directions using normalized (DNS) data as input:
figg, axx = plt.subplots(figsize=(12,6))
axx.loglog(freqs_NS_, psd_NS_, label='North-South')#, color='green')
axx.loglog(freqs_EW_, psd_EW_, label='East-West')#, color='blue')
axx.loglog(freqs_Z_, psd_Z_, label='Up-Down', alpha=0.8)#, color='magenta')
axx.set_xlabel('Frequency (Hz)')
axx.set_ylabel('PSD')
axx.set_ylim(bottom=10**-9)
#axx.set_xlim(right=500)
axx.set_title('Power Spectral Density For NS, EW, & Z Components')
plt.grid(True, which='both', alpha=0.5)
axx.legend(loc='upper left')
plt.show()

***
#### Horizontal-to-Vertical-Spectral-Ratio (HVSR):
- Method 1: FFT-based HVSR
    - Based on original version of FFT code
    - Relies on Savitzky-Golay filter for smoothing
- Method 2: Segmented FFT-based HVSR
    - Based on plot_averaged_fft approach
    - Gave worst curve out of the three methods
- Method 3: PSD-based HVSR (Welch's Method)
    - Can also use Savitzky-Golay filter here for smoother result, but less necessary than in Method 1

In [ ]:
# Method 1: FFT-based HVSR
def H_V_ratio_fft(data_ns, column_ns, data_ew, column_ew, data_z, column_z, sr, apply_window=True, freq_min=0.1, freq_max=None, combine_method='rms'):
    """Compute Horizontal-to-Vertical Spectral Ratio (HVSR) using FFT.
    data_ns: pandas DataFrame containing North-South component
    column_ns: column name for NS component, string
    data_ew: pandas DataFrame containing East-West component
    column_ew: column name for EW component, string
    data_z: pandas DataFrame containing vertical component
    column_z: column name for Z component, string
    sr: sampling rate in Hz
    apply_window: whether to apply partial Hann taper, bool (default True)
    freq_min: minimum frequency to include in output, Hz (default 0.1)
    freq_max: maximum frequency to include, Hz (default 80% of Nyquist)
    combine_method: how to combine NS and EW horizontals:
                    'rms' for geometric mean: sqrt((ns**2 + ew**2) / 2)
                    'vector' for vector magnitude: sqrt(ns**2 + ew**2)

    Returns:
    freqs: frequency array (Hz), masked to [freq_min, freq_max]
    hvsr: H/V spectral ratio, masked to [freq_min, freq_max]"""
    n = len(data_ns)

    if freq_max is None:
        freq_max = 0.8 * (sr / 2) # 80% of Nyquist

    # Apply window if requested
    if apply_window:
        windowed_data_ns = apply_hann_window1(data_ns, column_ns, taper_fraction=0.1).values
        windowed_data_ew = apply_hann_window1(data_ew, column_ew, taper_fraction=0.1).values
        windowed_data_z  = apply_hann_window1(data_z,  column_z,  taper_fraction=0.1).values
        coherent_gain = calculate_coherent_gain(data_ns[column_ns], taper_fraction=0.1)
    else:
        windowed_data_ns = data_ns[column_ns].values
        windowed_data_ew = data_ew[column_ew].values
        windowed_data_z  = data_z[column_z].values
        coherent_gain= 1.0

    #Compute FFTs
    fft_ns = scipy.fft.rfft(windowed_data_ns)
    fft_ew = scipy.fft.rfft(windowed_data_ew)
    fft_z  = scipy.fft.rfft(windowed_data_z)
    freqs  = scipy.fft.rfftfreq(n, d=1/sr)

    # Compute magnitudes WITH coherent gain correction
    mag_ns = np.abs(fft_ns) / (n * coherent_gain)
    mag_ew = np.abs(fft_ew) / (n * coherent_gain)
    mag_z  = np.abs(fft_z)  / (n * coherent_gain)

    # Double AC components for one-sided spectrum 
    mag_ns[1:] *= 2.0
    mag_ew[1:] *= 2.0
    mag_z[1:]  *= 2.0

    # Undo doubling for Nyquist if even length
    if n % 2 == 0:
        mag_ns[-1] /= 2.0
        mag_ew[-1] /= 2.0
        mag_z[-1]  /= 2.0

    # Combine horizontal components
    if combine_method == 'rms':
        mag_horizontal = np.sqrt((mag_ns**2 + mag_ew**2) / 2)  # geometric mean (seismology standard)
    elif combine_method == 'vector':
        mag_horizontal = np.sqrt(mag_ns**2 + mag_ew**2)        # vector magnitude
    else:
        raise ValueError("combine_method must be 'rms' or 'vector'")

    # Apply frequency mask first, then compute threshold within that range:
    freq_mask = (freqs >= freq_min) & (freqs <= freq_max)

    # Compute division threshold within the masked frequency range only
    z_threshold = np.max(mag_z[freq_mask]) * 1e-8

    # Compute HVSR with threshold to avoid division by near-zero values
    hvsr = np.zeros_like(mag_horizontal)
    mask = mag_z > z_threshold
    hvsr[mask] = mag_horizontal[mask] / mag_z[mask]

    return freqs[freq_mask], hvsr[freq_mask]
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Get result for df data:
freqs1, hvsr1 = H_V_ratio_fft(df, 'NS_Normalized_Shifted', df, 'EW_Normalized_Shifted', df, 'Z_Normalized_Shifted', original_sampling_rate)
# Apply smoothing with Savitzky-Golay filter
hvsr1_smooth = savgol_filter(hvsr1, window_length=20481, polyorder=2) # window_length should be an odd integer
# add more filter passes as needed
hvsr1_smooth = savgol_filter(hvsr1_smooth, window_length=2001, polyorder=2)
# Find main peak in given frequency range:
peak_idx = np.argmax(hvsr1_smooth[(freqs1 > 10) & (freqs1 < 50)])
peak_freq = freqs1[(freqs1 > 10) & (freqs1 < 50)][peak_idx]
peak_value = hvsr1_smooth[(freqs1 > 10) & (freqs1 < 50)][peak_idx]
print(f"Full data peak frequency: {peak_freq} Hz")
print(f"Full data peak H/V: {peak_value}")

# Plot:
plt.figure(figsize=(12, 6))
plt.semilogx(freqs1, hvsr1_smooth, label="H/V From df Data")
plt.xlabel('Frequency (Hz)')
plt.ylabel('H/V Ratio')
plt.title('Horizontal-to-Vertical Spectral Ratio (HVSR)')
plt.grid(True, which="both", alpha=0.5)
plt.axvline(x=peak_freq, linestyle=":", color='orange', label=f'Peak freq = {peak_freq:.2f}') 
plt.legend(loc='upper left')
plt.show()

In [ ]:
# Option 2: Segmented FFT-based HVSR
# Does not give very smooth curve
def H_V_ratio_fft_segmented(data_ns, column_ns, data_ew, column_ew, data_z, column_z, sr, segment_size=1000, overlap=0.5,
                              freq_min=0.1, freq_max=None, combine_method='rms'):
    """Compute HVSR using segmented FFT averaging (matches plot_averaged_fft approach).
    Computes H/V ratio per segment, then RMS-averages across segments.
    This follows the SESAME guideline approach of averaging ratios, not spectra.
    
    data_ns: pandas DataFrame containing North-South component
    column_ns: column name for NS component, string
    data_ew: pandas DataFrame containing East-West component
    column_ew: column name for EW component, string
    data_z: pandas DataFrame containing vertical component
    column_z: column name for Z component, string
    sr: sampling rate in Hz
    segment_size: length of each FFT segment (controls frequency resolution), int
    overlap: fractional overlap between segments, float (default 0.5 = 50%)
    freq_min: minimum frequency to include in output, Hz (default 0.1)
    freq_max: maximum frequency to include, Hz (default 80% of Nyquist)
    combine_method: how to combine NS and EW horizontals:
                    'rms' for geometric mean: sqrt((ns² + ew²) / 2) 
                    'vector' for vector magnitude: sqrt(ns² + ew²)
    Returns
    freqs: frequency array (Hz), masked to [freq_min, freq_max]
    hvsr: RMS-averaged H/V spectral ratio
    hvsr_std : standard deviation across segments (measure of stability)"""
    if freq_max is None:
        freq_max = 0.8 * (sr / 2)

    # Extract signal arrays
    x_ns = data_ns[column_ns].values
    x_ew = data_ew[column_ew].values
    x_z  = data_z[column_z].values
    N    = len(x_ns)

    # Segment parameters
    step_size = int(segment_size * (1 - overlap))
    num_segments = (N - segment_size) // step_size + 1
    n_freqs= segment_size // 2 + 1

    # Hann window for each segment
    window = np.hanning(segment_size)
    coherent_gain = np.mean(window)  # for amplitude correction

    # Accumulate squared HVSR per segment (for RMS averaging)
    hvsr_sq_accum = np.zeros(n_freqs)
    hvsr_segments = []  # store per-segment ratios for std calculation

    for i in range(num_segments):
        start = i * step_size
        end   = start + segment_size

        # Extract and window each segment
        seg_ns = x_ns[start:end] * window
        seg_ew = x_ew[start:end] * window
        seg_z  = x_z[start:end]  * window

        # Compute FFT for each component
        fft_ns = scipy.fft.rfft(seg_ns, segment_size)
        fft_ew = scipy.fft.rfft(seg_ew, segment_size)
        fft_z  = scipy.fft.rfft(seg_z,  segment_size)

        # Compute magnitudes with coherent gain correction
        mag_ns = np.abs(fft_ns) / (segment_size * coherent_gain)
        mag_ew = np.abs(fft_ew) / (segment_size * coherent_gain)
        mag_z  = np.abs(fft_z)  / (segment_size * coherent_gain)

        # Double AC components for one-sided spectrum
        mag_ns[1:] *= 2.0
        mag_ew[1:] *= 2.0
        mag_z[1:]  *= 2.0

        # Undo doubling for Nyquist bin if even
        if segment_size % 2 == 0:
            mag_ns[-1] /= 2.0
            mag_ew[-1] /= 2.0
            mag_z[-1]  /= 2.0

        # Combine horizontal components
        if combine_method == 'rms':
            mag_horizontal = np.sqrt((mag_ns**2 + mag_ew**2) / 2)
        elif combine_method == 'vector':
            mag_horizontal = np.sqrt(mag_ns**2 + mag_ew**2)
        else:
            raise ValueError("combine_method must be 'rms' or 'vector'")

        # Compute per-segment HVSR with threshold guard
        z_threshold = np.max(mag_z) * 1e-8
        hvsr_seg = np.zeros(n_freqs)
        valid = mag_z > z_threshold
        hvsr_seg[valid] = mag_horizontal[valid] / mag_z[valid]

        # Accumulate squared ratios for RMS averaging
        hvsr_sq_accum += hvsr_seg**2
        hvsr_segments.append(hvsr_seg)

    # RMS average across segments
    hvsr = np.sqrt(hvsr_sq_accum / num_segments)

    # Standard deviation across segments (stability indicator)
    hvsr_stack = np.array(hvsr_segments)  # shape: (num_segments, n_freqs)
    hvsr_std   = np.std(hvsr_stack, axis=0)

    # Frequency bins
    freqs = scipy.fft.rfftfreq(segment_size, d=1/sr)

    # Apply frequency mask
    freq_mask = (freqs >= freq_min) & (freqs <= freq_max)

    return freqs[freq_mask], hvsr[freq_mask], hvsr_std[freq_mask]
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# freqs_segmented1, HV_segmented1, HV_std1 = H_V_ratio_fft_segmented(df, 'NS_Normalized_Shifted', df, 'EW_Normalized_Shifted', df, 
#                                             'Z_Normalized_Shifted', original_sampling_rate, segment_size=250, freq_max=100)
# # Try SG filter here too?
# #HV_segmented1_smooth = savgol_filter(HV_segmented1, window_length=5, polyorder=2) # window_length should be an odd integer
# # Find main peak:
# peak_idx_s = np.argmax(HV_segmented1[(freqs_segmented1 > 10) & (freqs_segmented1 < 50)])
# peak_freq_s = freqs_segmented1[(freqs_segmented1 > 10) & (freqs_segmented1 < 50)][peak_idx_s]
# peak_value_s = HV_segmented1[(freqs_segmented1 > 10) & (freqs_segmented1 < 50)][peak_idx_s]
# print(f"Full data peak frequency: {peak_freq_s} Hz")
# print(f"Full data peak H/V: {peak_value_s}")

# # Plot:
# plt.figure(figsize=(12, 6))
# plt.semilogx(freqs_segmented1, HV_segmented1, label="H/V From df Data")
# plt.xlabel('Frequency (Hz)')
# plt.ylabel('H/V Ratio')
# plt.title('Horizontal-to-Vertical Spectral Ratio (HVSR)')
# plt.grid(True, which="both", alpha=0.5)
# plt.axvline(x=peak_freq_s, linestyle=":", color='orange', label=f'Peak freq = {peak_freq_s:.2f}') 
# plt.legend(loc='upper left')
# plt.show()

In [ ]:
# Option 3: PSD-based HVSR (Welch's method)
def H_V_ratio_psd(data_ns, column_ns, data_ew, column_ew, data_z, column_z, sr, segment_size=1000, freq_min=0.1, freq_max=None, combine_method='rms', 
                  window='hann'):
    """Compute Horizontal-to-Vertical Spectral Ratio (HVSR) using Welch PSD. Welch averaging reduces noise and produces a smoother ratio curve.
    data_ns: pandas DataFrame containing North-South component
    column_ns: column name for NS component, string
    data_ew: pandas DataFrame containing East-West component
    column_ew: column name for EW component, string
    data_z: pandas DataFrame containing vertical component
    column_z: column name for Z component, string
    sr: sampling rate in Hz
    segment_size : length of each Welch segment (controls freq resolution), int
    freq_min: minimum frequency to include in output, Hz (default 0.1)
    freq_max: maximum frequency to include, Hz (default 80% of Nyquist)
    combine_method: how to combine NS and EW horizontals:
                    'rms' for geometric mean: sqrt((psd_ns + psd_ew) / 2)
                    'vector' for sum: psd_ns + psd_ew
    window: window function for Welch (default 'hann')

    Returns
    freqs: frequency array (Hz), masked to [freq_min, freq_max]
    hvsr: H/V spectral ratio, masked to [freq_min, freq_max]
    psd_ns, psd_ew, psd_z : individual component PSDs"""
    if freq_max is None:
        freq_max = 0.8 * (sr / 2)

    overlap = segment_size // 2  # 50% overlap

    #Compute Welch PSD for each component
    freqs, psd_ns = welch(data_ns[column_ns].values, fs=sr, nperseg=segment_size, noverlap=overlap, window=window)
    freqs, psd_ew = welch(data_ew[column_ew].values, fs=sr, nperseg=segment_size, noverlap=overlap, window=window)
    freqs, psd_z  = welch(data_z[column_z].values,   fs=sr, nperseg=segment_size, noverlap=overlap, window=window)

    # Combine horizontal PSDs
    # Note: working in power domain, so RMS = sqrt(mean(power))
    if combine_method == 'rms':
        psd_horizontal = np.sqrt((psd_ns + psd_ew) / 2) # geometric mean amplitude
        psd_vertical   = np.sqrt(psd_z)
    elif combine_method == 'vector':
        psd_horizontal = np.sqrt(psd_ns + psd_ew) # vector amplitude
        psd_vertical   = np.sqrt(psd_z)
    else:
        raise ValueError("combine_method must be 'rms' or 'vector'")

    # Apply frequency mask first
    freq_mask = (freqs >= freq_min) & (freqs <= freq_max)

    # Compute threshold within masked range only
    z_threshold = np.max(psd_vertical[freq_mask]) * 1e-8

    # Compute HVSR
    hvsr = np.zeros_like(psd_horizontal)
    mask = psd_vertical > z_threshold
    hvsr[mask] = psd_horizontal[mask] / psd_vertical[mask]

    return freqs[freq_mask], hvsr[freq_mask], psd_ns[freq_mask], psd_ew[freq_mask], psd_z[freq_mask]
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
freqs1_psd, hvsr1_psd, _, _, _ = H_V_ratio_psd(df, 'NS_Normalized_Shifted', df, 'EW_Normalized_Shifted', df, 'Z_Normalized_Shifted', 
                                               original_sampling_rate, segment_size=500)
# Try SG filter here too?
hvsr1_psd_smooth = savgol_filter(hvsr1_psd, window_length=13, polyorder=2) # window_length should be an odd integer
# Find main peak:
peak_idx_psd = np.argmax(hvsr1_psd_smooth[(freqs1_psd > 10) & (freqs1_psd < 50)])
peak_freq_psd = freqs1_psd[(freqs1_psd > 10) & (freqs1_psd < 50)][peak_idx_psd]
peak_value_psd = hvsr1_psd_smooth[(freqs1_psd > 10) & (freqs1_psd < 50)][peak_idx_psd]
print(f"Full data peak frequency: {peak_freq_psd} Hz")
print(f"Full data peak H/V: {peak_value_psd}")

# Plot:
plt.figure(figsize=(12, 6))
plt.semilogx(freqs1_psd, hvsr1_psd_smooth, label="H/V From df Data")
plt.xlabel('Frequency (Hz)')
plt.ylabel('H/V Ratio')
plt.title('Horizontal-to-Vertical Spectral Ratio (HVSR)')
plt.grid(True, which="both", alpha=0.5)
plt.axvline(x=peak_freq_psd, linestyle=":", color='orange', label=f'Peak freq = {peak_freq_psd:.2f}') 
plt.legend(loc='upper left')
plt.show()

In [ ]:
#HVSR Comparison Plot:
fig, ax = plt.subplots(figsize=(12,6))
ax.semilogx(freqs1, hvsr1_smooth, label="FFT-based H/V")
#ax.semilogx(freqs_segmented1, HV_segmented1, label="Segmented FFT-based H/V") # The other 2 results seem better
ax.semilogx(freqs1_psd, hvsr1_psd_smooth, label="PSD-based H/V")
ax.axvline(x=peak_freq, linestyle=":", color='purple', label=f'FFT-based peak freq: {peak_freq:.2f} Hz') 
#plt.axvline(x=peak_freq_s, linestyle=":", color='skyblue', label=f'Segmented FFT-based peak freq: {peak_freq_s:.2f}') 
ax.axvline(x=peak_freq_psd, linestyle=":", color='green', label=f'PSD-based peak freq: {peak_freq_psd:.2f} Hz') 
ax.set_ylabel("H/V")
ax.set_xlabel("Frequency (Hz)")
ax.legend(loc='upper left')
ax.set_title("Comparison HVSR")
ax.set_xlim(2,200)
plt.grid(True, which='both', alpha=0.5)
plt.show()

#### Interpreting Horizontal-to-Vertical Spectral Ratios
- Peaks should occur at fundamental and higher order resonance frequencies
    - (fundamental frequency = lowest resonant frequency in a system)
- Typically, the fundamental resonance frequency is the first/lowest value indicated by the highest amplitude peak on the H/V frequency spectrum
- H/V = 1 indicates that the horizontal and vertical motions are equal
- H/V < 1 indicates that horizontal motion is weaker than vertical motion
- H/V > 1 indicates that horizontal motion is stronger than vertical motion

### Plotting H/V Ratio From Grilla .txt File:

In [ ]:
# Trying to use H/V .txt file downloaded from Grilla:
column_names = ['freq.', 'H/V']
dtf = pd.read_table('Tyendinaga_HV_ratio.asc', delimiter=r'\s+', encoding='latin-1', skiprows=31, names=column_names, usecols=column_names)
#dtf.head().style

In [ ]:
plt.figure(figsize=(12,6))
plt.plot(dtf['freq.'], dtf['H/V'], label="H/V Ratio from Grilla Data File")
plt.xlabel("Frequency (Hz)")
plt.ylabel("H/V Ratio")
plt.title("H/V Ratio from Grilla Data File")
plt.axvline(x=30.63, color='red', linestyle='--', label="30.63 Hz")
plt.legend()
plt.grid(True, which='both', alpha=0.5)
plt.xscale('log')
plt.show()

In [ ]:
# Remove unneeded columns: NS, EW, Z, nsL, ewL, zL, aY, aX, aZ, NS_Detrended, EW_Detrended, Z_Detrended, NS_Normalized, EW_Normalized, Z_Normalized
# Will be left with DataFrame containing time (s) and the Normalized_Shifted data for NS, EW, and Z components
df_updated = df.drop(['NS', 'EW', 'Z', 'nsL', 'ewL', 'zL', 'aY', 'aX', 'aZ', 'NS_Detrended', 'EW_Detrended', 'Z_Detrended', 'NS_Normalized', 
                      'EW_Normalized', 'Z_Normalized'], axis=1)
# df_updated.head()

### Convert Cell Outputs to PDF:
- can currently download as .html file, which when opened in a browser can be saved as a pdf using print (Ctrl+P) -> save to pdf <br>
    - using code to directly download as .pdf file is not working currently
    - if opened with Jupyter Notebook, can save full file as .pdf directly by going to File -> Save and Export Notebook As -> PDF

In [ ]:
# Save notebook as .html file
# def save_notebook(notebook_name, output_name=None):
#     """Save Jupyter notebook outputs and markdown to .html file (excludes code cells)
#     notebook_name : str, name of the notebook file (e.g., 'notebook.ipynb')
#     output_name : str, name for the output file (optional). If None, uses the notebook name with .html extension"""
    
#     # Set output file name
#     if output_name is None:
#         output_name = notebook_name.replace('.ipynb', '.html')
    
#     # Check if notebook file exists
#     if not os.path.exists(notebook_name):
#         raise FileNotFoundError(f"Notebook file '{notebook_name}' not found")
            
#     result = subprocess.run(['jupyter', 'nbconvert', '--to', 'html', '--no-input',  # Exclude code cells
#     notebook_name, '--output', output_name.replace('.html', '')], capture_output=True, text=True)
            
#     if result.returncode == 0: # return code of zero indicates successful execution 
#         print(f"Created HTML file: {output_name}")
#         print("Open this in your browser and use Ctrl+P --> Save as PDF")
#         print("(The HTML file contains only outputs and markdown, no code cells)")
#     else:
#         print("Error:")
#         print(result.stderr) # standard error

# Example usage:
# save_notebook('notebook.ipynb')
# Or specify custom output name:
# save_notebook('notebook.ipynb', 'output_notebook.html')